In [1]:
import rowordnet as rwn
wn = rwn.RoWordNet()

c:\Users\Tudor\anaconda3\envs\nlp-processing-env\Lib\site-packages\rowordnet\rowordnet.py:41: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
words = {
    "nouns": ["manevră", "ușă", "sticlă"],
    "verbs": ["dărâma", "citi", "conduce"],
    "adjs": ["nobil", "mândru"]
}

In [3]:

def get_synset_main_info(synset_id):
    synset_obj = wn.synset(synset_id)
    pno = synset_obj.sentiwn
    info = [
        f"Synset ID '{synset_id}'",
        f"Literal {synset_obj.literals}; Literal Senses: {synset_obj.literals_senses}; Definition: {synset_obj.definition}",
        f"PNO (Positive: {pno[0]}, Negative: {pno[1]}, Objective: {pno[2]})"
    ]
    return "\n".join(info)


def get_synset_relations(synset_id):
    outbound_relations = wn.outbound_relations(synset_id)
    if(len(outbound_relations) == 0):
        return f"No outbound relations for synset {synset_id}"
        
    lines = ["+ Relations"]
    for outbound_relation in outbound_relations:
        target_synset_id = wn.synset(outbound_relation[0])
        relation = outbound_relation[1]
        lines.append(f"  - Relation [{relation}] -> to synset {target_synset_id.literals} ({target_synset_id.definition})")
    return "\n".join(lines)

#get all the hypernyms of the sysnet until the root of the hierarchy is reached
def get_hypernyms(synset_id):
    current_synset_id = synset_id
    hypernyms = []
    while True:
        outbound_relations = wn.outbound_relations(current_synset_id)
        hypernym = next(filter(lambda x: x[1] == "hypernym", outbound_relations), None)
        if hypernym is None:
            break
        synset = wn.synset(hypernym[0])
        if(len(synset.literals) > 0):
            hypernyms.insert(0, synset.literals[0])
        current_synset_id = hypernym[0]
    return hypernyms

def get_hyponyms(synset_id):

    outbound_relations = wn.outbound_relations(synset_id)
    hyponym_relations = filter(lambda x: x[1] == "hyponym", outbound_relations)
    hyponyms = []

    for relation in hyponym_relations:
        target_synset = wn.synset(relation[0])
        hyponyms.append(target_synset.literals[0] if len(target_synset.literals) > 0 else target_synset.definition)

    return hyponyms
    
def get_hierarchy_tree(word, synset_id, pos_type):
    if pos_type not in [rwn.Synset.Pos.NOUN, rwn.Synset.Pos.VERB]:
        return ""
    hypernyms = get_hypernyms(synset_id)
    hyponyms = get_hyponyms(synset_id)

    tree_lines = ["Hierarchy tree:"]
    if not hypernyms:
        tree_lines.append(f"[{word}]")
        indent = ""
    else:
        tree_lines.append(hypernyms[0])
        curr_indent = ""
        for h in hypernyms[1:]:
            curr_indent += "   "
            tree_lines.append(f"{curr_indent}|_ {h}")
        curr_indent += "   "
        tree_lines.append(f"{curr_indent}|_ [{word}]")
        indent = curr_indent

    if not hyponyms:
        tree_lines.append(f"{indent}   |_ (none)")
    else:
        for hy in hyponyms:
            tree_lines.append(f"{indent}   |_ {hy}")
    
    return "\n".join(tree_lines)



In [4]:
def double_print(text):
    print(text)
    f.write(text + "\n")

def get_pos(pos):
    if pos == "nouns":
        return rwn.Synset.Pos.NOUN
    elif pos == "verbs":
        return rwn.Synset.Pos.VERB
    elif pos == "adjs":
        return rwn.Synset.Pos.ADJECTIVE
    else:
        return None
    
with open('rowordnet_output.txt', 'w', encoding='utf-8') as f:

    for pos, word_list in words.items():
        double_print(f"Part of Speech: {pos}")
        p = get_pos(pos)

        for word in word_list:
            synset_ids = wn.synsets(literal=word, pos=p, strict=True)
            double_print("Word: '{}'".format(word))
            for synset_id in synset_ids:
                double_print(get_synset_main_info(synset_id))
                double_print(get_synset_relations(synset_id))

                tree = get_hierarchy_tree(word, synset_id, p)
                if tree:
                    double_print(tree)
                double_print("\n")
            double_print("\n")
        double_print("\n")
        

Part of Speech: nouns
Word: 'manevră'
Synset ID 'ENG30-00959992-n'
Literal ['manevră']; Literal Senses: ['2']; Definition: Pregătire tactică a unei armate sau a unei flote, în condiții asemănătoare cu cele de război
PNO (Positive: 0.0, Negative: 0.0, Objective: 1.0)
+ Relations
  - Relation [hypernym] -> to synset ['operație', 'operațiune'] (Acțiune militară de mare amploare, în vederea realizării unui plan strategic sau a sarcinilor subordonate acesteia)
  - Relation [part_holonym] -> to synset ['instrucție'] (Pregătire a ostașilor în vederea însușirii teoriei și practicii militare.)
  - Relation [domain_TOPIC] -> to synset ['armată', 'forță_armată', 'trupă'] (Totalitatea forțelor militare ale unui stat; oaste, oștire, armie)
  - Relation [near_eng_derivat] -> to synset ['face_manevre'] (a face mișcări de trupe în așa fel încât să obțină un avantaj pe câmpul de luptă)
Hierarchy tree:
entitate
   |_ abstractizare
      |_ trăsătură_psihologică
         |_ event
            |_ faptă
   